# Comparaison Finetuning (Prompt 1) vs Few-shot KATE

Compare le **meilleur modèle finetuné** (Prompt 1) et le **few-shot KATE** pour analyser dans quelle mesure les deux approches sont **similaires** (mêmes erreurs / mêmes succès) ou au contraire **complémentaires** (l’un a raison quand l’autre se trompe).

## 1. Configuration et chemins

In [ ]:
from pathlib import Path

NLI4CT_ROOT = Path(".").resolve()
if not (NLI4CT_ROOT / "results").exists():
    NLI4CT_ROOT = NLI4CT_ROOT / "NLI4CT"
if not (NLI4CT_ROOT / "results").exists():
    raise FileNotFoundError("Dossier results/ introuvable.")

RESULTS_DIR_P1 = NLI4CT_ROOT / "results" / "Prompt 1"
RESULTS_DIR_KATE = NLI4CT_ROOT / "results" / "Fewshot_Kate"
FIGURES_DIR = NLI4CT_ROOT / "results" / "compare_figures_ft_vs_kate"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

GOLD_TEST_JSON = NLI4CT_ROOT / "Gold_test.json"
GOLD_JSONL_P1 = RESULTS_DIR_P1 / "Gold_test_formatted_old_prompt.jsonl" if (RESULTS_DIR_P1 / "Gold_test_formatted_old_prompt.jsonl").exists() else RESULTS_DIR_P1 / "Gold_test_formatted.jsonl"

csv_ft_1 = list(RESULTS_DIR_P1.glob("pred_ft_*.csv"))
csv_kate = list(RESULTS_DIR_KATE.glob("pred_fewshot_KATE*.csv"))
CSV_FT_1 = csv_ft_1[0] if csv_ft_1 else None
CSV_KATE = csv_kate[0] if csv_kate else None

print("Prompt 1 Finetuné:", CSV_FT_1)
print("KATE Few-shot:", CSV_KATE)
print("Figures:", FIGURES_DIR)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

## 2. Chargement des données (P1 Finetuné + KATE)

In [ ]:
def extract_premise_hypothesis(text):
    if "HYPOTHESIS:" not in text:
        return text[:800] if len(text) > 800 else text, text[:500] if len(text) > 500 else text
    before, after = text.split("HYPOTHESIS:", 1)
    premise = before.replace("PREMISE:", "").strip()
    for sep in (".\n\nAnswer", "?\n\nAnswer", "? Answer", "\n\nAnswer"):
        if sep in after: after = after.split(sep)[0]
    return premise, after.strip()

assert CSV_FT_1 and CSV_KATE, "Il manque un CSV (finetuné ou KATE)."

df_ft_1 = pd.read_csv(CSV_FT_1)
df_kate = pd.read_csv(CSV_KATE)
for df in (df_ft_1, df_kate):
    if df['is_correct'].dtype == object:
        df['is_correct'] = df['is_correct'].apply(lambda x: str(x).strip().lower() == 'true')

meta_list = []
with open(GOLD_JSONL_P1, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if not line.strip(): continue
        obj = json.loads(line)
        msgs = obj.get("messages", [])
        if len(msgs) < 2: continue
        user_content = ""
        for m in msgs:
            if m.get("role") == "user": user_content = m.get("content", "")
        prem, stmt = extract_premise_hypothesis(user_content)
        meta_list.append({'index': idx, 'premise_jsonl': prem, 'statement': stmt})
df_meta = pd.DataFrame(meta_list)
df_ft_1 = df_ft_1.merge(df_meta, on='index', how='left')
df_kate = df_kate.merge(df_meta, on='index', how='left')
for df in (df_ft_1, df_kate):
    df['premise'] = df['premise_jsonl'].astype(str)
    df['hypothesis'] = df['statement'].astype(str)
    df['prompt_len'] = df['premise'].str.len() + df['hypothesis'].str.len()

if GOLD_TEST_JSON.exists():
    with open(GOLD_TEST_JSON, 'r', encoding='utf-8') as f: gold = json.load(f)
    keys = list(gold.keys())
    if len(keys) >= len(df_ft_1):
        type_sec = pd.DataFrame([{'index': i, 'type': gold[keys[i]].get('Type','N/A'), 'section_id': gold[keys[i]].get('Section_id','N/A')} for i in range(len(keys))])
        for df in (df_ft_1, df_kate):
            df['type'] = type_sec.set_index('index').reindex(df['index'])['type'].values
            df['section_id'] = type_sec.set_index('index').reindex(df['index'])['section_id'].values

print("P1 Finetuné:", len(df_ft_1), "| KATE Few-shot:", len(df_kate))

## 3. Statistiques globales

In [ ]:
acc_dict = {
    'P1 Finetuné': [df_ft_1['is_correct'].mean()],
    'KATE Few-shot': [df_kate['is_correct'].mean()],
}
acc_glob = pd.DataFrame(acc_dict)
display(acc_glob)
fig, ax = plt.subplots(figsize=(8, 4))
cols = list(acc_glob.columns)
x = np.arange(len(cols))
ax.bar(x, [acc_glob[c].iloc[0] for c in cols], color=['#3498db', '#9b59b6'])
ax.set_xticks(x)
ax.set_xticklabels(cols)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy globale — P1 Finetuné vs KATE Few-shot')
ax.set_ylim(0, 1)
for i, v in enumerate([acc_glob[c].iloc[0] for c in cols]):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '01_accuracy_globale.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Accord / désaccord — similarité vs complémentarité

In [ ]:
# Aligner sur index commun
common_idx = list(set(df_ft_1['index']) & set(df_kate['index']))
ok_p1 = df_ft_1.set_index('index').loc[common_idx, 'is_correct'].reindex(common_idx).fillna(False).values
ok_kate = df_kate.set_index('index').loc[common_idx, 'is_correct'].reindex(common_idx).fillna(False).values

acc = pd.DataFrame({'index': common_idx, 'P1_ft': ok_p1, 'KATE': ok_kate})
acc['pattern'] = acc['P1_ft'].astype(int).astype(str) + acc['KATE'].astype(int).astype(str)
pattern_labels = {'11': 'Les 2 corrects', '00': 'Les 2 en erreur', '10': 'P1✓ KATE✗', '01': 'P1✗ KATE✓'}
counts = acc['pattern'].value_counts().reindex(list(pattern_labels.keys())).fillna(0).astype(int)
tab_accord = pd.DataFrame({'Pattern': [pattern_labels.get(p, p) for p in counts.index], 'Effectif': counts.values}, index=counts.index)
display(tab_accord)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(tab_accord))
colors_bar = ['#2ecc71' if '2 corrects' in str(l) else '#e74c3c' if '2 en erreur' in str(l) else '#f39c12' for l in tab_accord['Pattern']]
ax.bar(x, tab_accord['Effectif'], color=colors_bar)
ax.set_xticks(x)
ax.set_xticklabels(tab_accord['Pattern'], rotation=25, ha='right')
ax.set_ylabel("Nombre d'exemples")
ax.set_title('Accord P1 Finetuné / KATE Few-shot — similarité vs complémentarité')
for i, v in enumerate(tab_accord['Effectif']): ax.text(i, v + 3, str(int(v)), ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '04_accord_ft_vs_kate.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analyse par type (Single vs Comparison) et par section

In [ ]:
# Accuracy par type et par section (si colonnes présentes)
if 'type' in df_ft_1.columns:
    by_type = pd.DataFrame({
        'P1 Finetuné': df_ft_1.groupby('type')['is_correct'].mean(),
        'KATE Few-shot': df_kate.groupby('type')['is_correct'].mean(),
    })
    display(by_type)
if 'section_id' in df_ft_1.columns:
    by_sec = pd.DataFrame({
        'P1 Finetuné': df_ft_1.groupby('section_id')['is_correct'].mean(),
        'KATE Few-shot': df_kate.groupby('section_id')['is_correct'].mean(),
    })
    display(by_sec.head(15))

## 6. Cas où les 2 se trompent vs cas où les 2 ont raison

In [ ]:
two_wrong = acc[acc['pattern']=='00']
two_ok = acc[acc['pattern']=='11']
p1_ok_kate_ko = acc[acc['pattern']=='10']
p1_ko_kate_ok = acc[acc['pattern']=='01']

print('Les 2 en erreur:', len(two_wrong))
print('Les 2 corrects:', len(two_ok))
print('P1✓ KATE✗ (finetuning seul a raison):', len(p1_ok_kate_ko))
print('P1✗ KATE✓ (few-shot seul a raison):', len(p1_ko_kate_ok))

if len(two_wrong) > 0 and 'section_id' in df_ft_1.columns:
    detail = df_ft_1[df_ft_1['index'].isin(two_wrong['index'])][['type','section_id']]
    print('\nRépartition des "2 en erreur" par section:')
    display(detail['section_id'].value_counts().head(10))
if len(two_ok) > 0 and 'section_id' in df_ft_1.columns:
    detail_ok = df_ft_1[df_ft_1['index'].isin(two_ok['index'])][['type','section_id']]
    print('\nRépartition des "2 corrects" par section:')
    display(detail_ok['section_id'].value_counts().head(10))

## 7. Exemples pour analyse (5 par cas)

In [ ]:
np.random.seed(42)
N_EX = 5
df_base = df_ft_1[['index','premise','hypothesis','true_label','type','section_id']].copy()
df_base['pred_P1_ft'] = df_base['index'].map(df_ft_1.set_index('index')['predicted_label'])
df_base['pred_KATE'] = df_base['index'].map(df_kate.set_index('index')['predicted_label'])

def show_examples(acc_subset, title):
    if len(acc_subset) == 0:
        print(f"{title}: aucun exemple.")
        return
    indices = acc_subset['index'].values
    sample = np.random.choice(indices, size=min(N_EX, len(indices)), replace=False)
    sub = df_base[df_base['index'].isin(sample)].set_index('index').loc[sample]
    print(f"\n{'='*60}")
    print(f"{title} ({len(indices)} ex., {len(sample)} affichés)")
    print('='*60)
    for idx in sample:
        row = sub.loc[idx]
        prem = (row['premise'][:200] + '...') if len(str(row['premise'])) > 200 else row['premise']
        hyp = (row['hypothesis'][:150] + '...') if len(str(row['hypothesis'])) > 150 else row['hypothesis']
        print(f"\n--- Index {idx} (type={row['type']}, section={row['section_id']}) ---")
        print(f"Vrai: {row['true_label']} | P1 FT: {row['pred_P1_ft']} | KATE: {row['pred_KATE']}")
        print(f"Premise: {prem}")
        print(f"Hypothesis: {hyp}")

show_examples(two_wrong, '1. Les 2 se trompent')
show_examples(two_ok, '2. Les 2 ont raison')
show_examples(p1_ok_kate_ko, '3. P1 FT a raison, KATE se trompe')
show_examples(p1_ko_kate_ok, '4. KATE a raison, P1 FT se trompe')

## 8. Synthèse

In [ ]:
lines = [
    '# Synthèse Finetuning (P1) vs Few-shot KATE',
    '',
    '## Accuracy',
    f"P1 Finetuné: {df_ft_1['is_correct'].mean():.2%}",
    f"KATE Few-shot: {df_kate['is_correct'].mean():.2%}",
    '',
    '## Accord / complémentarité',
    f"Les 2 corrects: {len(two_ok)} | Les 2 en erreur: {len(two_wrong)}",
    f"P1✓ KATE✗: {len(p1_ok_kate_ko)} | P1✗ KATE✓: {len(p1_ko_kate_ok)}",
]
synth = '\n'.join(lines)
print(synth)
with open(FIGURES_DIR / 'synthese_ft_vs_kate.txt', 'w', encoding='utf-8') as f:
    f.write(synth)

## 9. Export des résultats détaillés

In [ ]:
export = df_ft_1[['index','premise','hypothesis','true_label','type','section_id']].copy()
export['P1_ft_correct'] = export['index'].map(df_ft_1.set_index('index')['is_correct'])
export['KATE_correct'] = export['index'].map(df_kate.set_index('index')['is_correct'])
export['pattern'] = export['index'].map(acc.set_index('index')['pattern'])
export.to_csv(FIGURES_DIR / 'error_analysis_ft_vs_kate.csv', index=False)
print('Exporté:', FIGURES_DIR / 'error_analysis_ft_vs_kate.csv')